# EDA — E-Commerce Fraud Data

**Dataset**: `Fraud_Data.csv`  
**Target**: `class` (1 = Fraud, 0 = Legitimate)  
**Objective**: Understand data quality, distributions, class imbalance, and fraud patterns.

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_fraud_data, load_ip_country, data_quality_report
from preprocessing import clean_fraud_data, merge_ip_to_country, engineer_fraud_features
from visualization import (
    plot_class_distribution,
    plot_numerical_distributions,
    plot_categorical_distributions,
    plot_bivariate,
    plot_fraud_by_country,
    plot_correlation_heatmap,
)

os.makedirs('../notebooks/plots', exist_ok=True)
print('✅ Imports successful')

## 1. Load & Inspect Raw Data

In [ ]:
fraud_raw = load_fraud_data('../data/raw/Fraud_Data.csv')
ip_country = load_ip_country('../data/raw/IpAddress_to_Country.csv')
fraud_raw.head()

In [ ]:
# Data quality report — before cleaning
quality_before = data_quality_report(fraud_raw, 'Fraud_Data (Raw)')

## 2. Data Cleaning

In [ ]:
fraud_clean = clean_fraud_data(fraud_raw)
quality_after = data_quality_report(fraud_clean, 'Fraud_Data (Cleaned)')
print(f'Rows removed: {len(fraud_raw) - len(fraud_clean):,}')

## 3. Class Imbalance Analysis

> **Why this matters**: Severe imbalance means standard accuracy is misleading.
> A model that always predicts "Legitimate" achieves high accuracy but catches zero fraud.
> We must use AUC-PR and F1-Score instead.

In [ ]:
fraud_counts = fraud_clean['class'].value_counts()
print('Class Distribution:')
print(f'  Legitimate (0): {fraud_counts.get(0, 0):,}  ({fraud_counts.get(0,0)/len(fraud_clean)*100:.2f}%)')
print(f'  Fraud      (1): {fraud_counts.get(1, 0):,}  ({fraud_counts.get(1,0)/len(fraud_clean)*100:.2f}%)')
print(f'  Imbalance ratio: {fraud_counts.get(0,1)/fraud_counts.get(1,1):.1f}:1')

plot_class_distribution(fraud_clean['class'], 'E-Commerce Fraud')

## 4. Univariate Distributions

In [ ]:
numerical_cols = ['purchase_value', 'age', 'ip_address']
plot_numerical_distributions(fraud_clean, numerical_cols, 'E-Commerce Fraud')

In [ ]:
cat_cols = ['source', 'browser', 'sex']
plot_categorical_distributions(fraud_clean, cat_cols, 'E-Commerce Fraud')

## 5. Bivariate Analysis (Feature vs. Target)

In [ ]:
plot_bivariate(fraud_clean, ['purchase_value', 'age'], 'class', 'E-Commerce Fraud')

In [ ]:
# Fraud rate by categorical variables
for col in ['source', 'browser', 'sex']:
    rates = fraud_clean.groupby(col)['class'].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    rates.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Fraud Rate by {col}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Fraud Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    plt.tight_layout()
    plt.savefig(f'../notebooks/plots/fraud_rate_by_{col}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Geolocation Integration

Map each `ip_address` to a country using range-based lookup via `pandas.merge_asof`.

In [ ]:
fraud_geo = merge_ip_to_country(fraud_clean, ip_country)
print(f'Countries identified: {fraud_geo["country"].nunique()}')
print(f'Unknown: {(fraud_geo["country"] == "Unknown").sum():,}')
fraud_geo[['ip_address', 'country', 'class']].head(10)

In [ ]:
plot_fraud_by_country(fraud_geo)

## 7. Feature Engineering

Derive behavioral and temporal features that capture fraud signals.

In [ ]:
fraud_featured = engineer_fraud_features(fraud_geo)

print('New features summary:')
fraud_featured[['time_since_signup', 'hour_of_day', 'day_of_week',
                 'transaction_count_per_user', 'transaction_velocity']].describe()

In [ ]:
# Distribution of time_since_signup by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, label, color in [(0, 'Legitimate', '#2ecc71'), (1, 'Fraud', '#e74c3c')]:
    subset = fraud_featured[fraud_featured['class'] == cls]['time_since_signup']
    axes[0].hist(subset.clip(upper=subset.quantile(0.99)), bins=50,
                 alpha=0.6, label=label, color=color)
axes[0].set_title('time_since_signup Distribution by Class', fontweight='bold')
axes[0].set_xlabel('Hours since signup')
axes[0].legend()

# Hour of day fraud rate
hourly_fraud = fraud_featured.groupby('hour_of_day')['class'].mean()
axes[1].bar(hourly_fraud.index, hourly_fraud.values, color='steelblue', edgecolor='white')
axes[1].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[1].set_xlabel('Hour of Day (0-23)')
axes[1].set_ylabel('Fraud Rate')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

plt.tight_layout()
plt.savefig('../notebooks/plots/time_features_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Processed Dataset

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
fraud_featured.to_csv('../data/processed/fraud_data_cleaned.csv', index=False)
print(f'✅ Saved fraud_data_cleaned.csv — shape: {fraud_featured.shape}')

## 9. EDA Summary

| Finding | Detail |
|---------|--------|
| Class imbalance | ~X% fraud — severe imbalance, requires SMOTE |
| Fraud timing | Short `time_since_signup` strongly associated with fraud |
| Browser/Source | Some channels show elevated fraud rates |
| Geolocation | Certain countries have significantly higher fraud rates |
| Age | Fraudsters tend to cluster in specific age ranges |

> **Next Step**: Feature engineering notebook → `feature-engineering.ipynb`